# K-Means Clustering Exercise

In this exercise you will build a complete K-means clustering pipeline from scratch, following the same steps as the tutorial.

**Scenario:** You work for a gym chain. The marketing team has collected data on members and wants to group them into meaningful segments so they can send targeted offers. Your job is to cluster the members and describe each group.

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Create and explore the dataset |
| 3 | Plot the raw data |
| 4 | Scale the features |
| 5 | Use the Elbow Method to find the best k |
| 6 | Calculate Silhouette Scores to confirm k |
| 7 | Fit the final K-means model |
| 8 | Visualise the clusters |
| 9 | Summarise and label each segment |
| 10 | Reflect and interpret |

**Complete every `# YOUR CODE HERE` section before moving on.**


Companion solution file: `KMeans_Exercise_Solutions.ipynb`

## Step 1 - Import Libraries

### Key Concept
Every machine learning project starts by importing the tools it needs. For K-means clustering we need:

| Library | Why we need it |
|---------|----------------|
| `numpy` | Numerical operations and array handling |
| `pandas` | Loading and working with tabular data |
| `matplotlib.pyplot` | Drawing charts |
| `sklearn.cluster.KMeans` | The K-means algorithm itself |
| `sklearn.preprocessing.StandardScaler` | Scaling features to equal range |
| `sklearn.metrics.silhouette_score` | Measuring cluster quality |

### What to do
Import all six items listed in the table above.

```python
# Guide — structure to follow:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
```

In [ ]:
# YOUR CODE HERE — import the six libraries listed above

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Libraries imported successfully.")

## Step 2 - Create and Explore the Dataset

### Key Concept
Before clustering you must understand your data. This means looking at:
- **Shape** — how many rows and columns?
- **Column names** — what features do we have?
- **Summary statistics** — what are the typical values, min and max?
- **Missing values** — are there any gaps that need handling?

The dataset below contains gym member data with two features:
- `visits_per_month` — how often the member visits each month
- `avg_session_minutes` — how long (in minutes) a typical visit lasts

There are four natural member types hidden in the data:
- Casual visitors (infrequent, short sessions)
- Social members (frequent, short sessions)
- Dedicated trainers (infrequent, long sessions)
- Elite members (frequent, long sessions)

### What to do
Run the data creation cell (do not modify it), then use pandas to:
1. Display the first 5 rows with `.head()`
2. Print the shape with `.shape`
3. Print summary statistics with `.describe()`
4. Check for missing values with `.isnull().sum()`

In [ ]:
# ── Data creation — DO NOT MODIFY THIS CELL ──────────────────────────
def make_gym_data(n=240):
    rng = np.random.default_rng(7)
    segments = [
        (4,  28, 0.25),   # casual: infrequent, short
        (16, 30, 0.25),   # social: frequent, short
        (5,  75, 0.25),   # dedicated: infrequent, long
        (18, 80, 0.25),   # elite: frequent, long
    ]
    rows = []
    for visit_mean, mins_mean, frac in segments:
        size    = int(n * frac)
        visits  = rng.normal(visit_mean, 1.5, size).clip(1, 25)
        minutes = rng.normal(mins_mean,  8,   size).clip(10, 120)
        rows.append(np.column_stack([visits, minutes]))
    data = np.vstack(rows)
    return pd.DataFrame(data, columns=['visits_per_month', 'avg_session_minutes'])

df = make_gym_data(n=240)
print("Dataset created.")

In [ ]:
# 1. Display the first 5 rows
# YOUR CODE HERE


In [ ]:
# 2. Print the shape (rows, columns)
# YOUR CODE HERE


In [ ]:
# 3. Print summary statistics
# YOUR CODE HERE


In [ ]:
# 4. Check for missing values
# YOUR CODE HERE


**Reflection:** How many members are in the dataset? What are the minimum and maximum values for each feature? Can you already guess how many natural groups might exist?

*Your answer:*

## Step 3 - Plot the Raw Data

### Key Concept
Always visualise your data before clustering. A scatter plot of the two features often reveals natural groupings that confirm your choice of k later on. At this stage we have no cluster labels — every point is the same colour.

### What to do
Create a scatter plot with:
- `visits_per_month` on the x-axis
- `avg_session_minutes` on the y-axis
- A descriptive title and axis labels

```python
# Guide — scatter plot template:
plt.figure(figsize=(8, 5))
plt.scatter(df['column_x'], df['column_y'],
            color='steelblue', alpha=0.6, edgecolors='white', s=60)
plt.xlabel('X Axis Label', fontsize=12)
plt.ylabel('Y Axis Label', fontsize=12)
plt.title('Your Title', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
```

In [ ]:
# YOUR CODE HERE — scatter plot of the raw unlabelled data


**Reflection:** Looking at the scatter plot, how many groups do you think exist in the data? Describe where they appear in the chart (e.g. top-left, bottom-right).

*Your answer:*

## Step 4 - Scale the Features

### Key Concept
K-means measures the **distance** between points to decide which cluster they belong to. If one feature has much larger values than another, it will dominate the distance calculation and produce biased clusters.

**Example from our data:**
- `visits_per_month` ranges from about 1 to 25
- `avg_session_minutes` ranges from about 10 to 120

Without scaling, `avg_session_minutes` would unfairly dominate. `StandardScaler` converts both features so they each have **mean = 0** and **standard deviation = 1**.

```
scaled_value = (original_value − mean) / std
```

### What to do
1. Create a `StandardScaler` object
2. Apply `.fit_transform()` to both columns of `df`
3. Store the result in `X_scaled`
4. Print the mean and std of each column before and after scaling

```python
# Guide:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[['column1', 'column2']])

# Check the result:
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(3)}")
print(f"Std  after scaling: {X_scaled.std(axis=0).round(3)}")
```

In [ ]:
# 1 & 2 & 3. Create scaler, fit and transform, store in X_scaled
# YOUR CODE HERE


# 4. Print mean and std BEFORE scaling
print("Before scaling:")
# YOUR CODE HERE


# 4. Print mean and std AFTER scaling
print("\nAfter scaling:")
# YOUR CODE HERE


**Reflection:** After scaling, what are the approximate mean and standard deviation of each feature? Why is it important that both features now have a similar scale?

*Your answer:*

## Step 5 - Find the Best k: The Elbow Method

### Key Concept
We need to choose **k** — the number of clusters — before running K-means. The **Elbow Method** helps us do this.

**How it works:**
1. Run K-means for k = 1, 2, 3 … 10
2. For each k, record the **WCSS** (Within-Cluster Sum of Squares), also called **inertia** — the total squared distance from every point to its cluster centroid
3. Plot WCSS against k
4. Look for the **"elbow"** — the point where the curve bends and improvement slows down

Adding more clusters always reduces WCSS, but beyond the elbow the improvement is small and not worth the added complexity.

```
km.inertia_   ← gives you the WCSS after fitting
```

### What to do
1. Create an empty list called `wcss`
2. Loop through k = 1 to 10
3. For each k: create a KMeans model, fit it to `X_scaled`, append `km.inertia_` to `wcss`
4. Plot WCSS vs k with a line chart
5. Mark where you think the elbow is with a vertical dashed line

```python
# Guide — loop template:
wcss    = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Guide — elbow plot template:
plt.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=YOUR_ELBOW_K, color='red', linestyle='--', label='Elbow')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
```

In [ ]:
# 1, 2 & 3. Calculate WCSS for k = 1 to 10
# YOUR CODE HERE
wcss    = []
k_range = range(1, 11)

for k in k_range:
    pass  # replace with your code

# Print the values
print("k  →  WCSS")
for k, w in zip(k_range, wcss):
    print(f"k={k:2d}  →  {w:.2f}")

In [ ]:
# 4 & 5. Plot the elbow curve and mark the elbow
# YOUR CODE HERE


**Reflection:** At which value of k does the elbow appear? What does the flattening of the curve after the elbow tell you?

*Your answer:*

## Step 6 - Confirm k with the Silhouette Score

### Key Concept
The **Silhouette Score** gives a second, independent opinion on the best k. It measures how well each point fits into its assigned cluster compared to the nearest neighbouring cluster.

| Score | Meaning |
|-------|----------|
| Close to **+1** | Point sits deep inside its cluster — excellent |
| Close to **0** | Point is near the boundary between two clusters |
| Close to **−1** | Point may belong to the wrong cluster |

We compute the **average** silhouette score across all points. **The k with the highest score is the best choice.**

> Important: silhouette score requires at least 2 clusters, so start from k = 2.

```python
# Guide:
from sklearn.metrics import silhouette_score

sil_scores  = []
k_range_sil = range(2, 11)

for k in k_range_sil:
    km     = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)          # fit and get cluster labels
    score  = silhouette_score(X_scaled, labels) # compute average score
    sil_scores.append(score)
```

In [ ]:
# Calculate silhouette scores for k = 2 to 10
# YOUR CODE HERE
sil_scores  = []
k_range_sil = range(2, 11)

for k in k_range_sil:
    pass  # replace with your code

# Print scores
print("k  →  Silhouette Score")
for k, s in zip(k_range_sil, sil_scores):
    print(f"k={k:2d}  →  {s:.4f}")

# Find and print the best k
best_k = k_range_sil[np.argmax(sil_scores)]
print(f"\nBest k by silhouette score: {best_k}")

In [ ]:
# Plot silhouette scores vs k
# YOUR CODE HERE
# Hint: plot k_range_sil on x-axis, sil_scores on y-axis
# Mark the best k with a vertical green dashed line


**Reflection:** Does the silhouette score agree with the elbow method? Which k does it suggest? Would you trust one method more than the other, or do you need both?

*Your answer:*

## Step 7 - Fit the Final Model

### Key Concept
Now that we have chosen k, we fit the final K-means model and assign every member to a cluster. The cluster label for each data point is stored in `km.labels_` after fitting, or returned directly by `.fit_predict()`.

After fitting, `km.cluster_centers_` gives the centroid coordinates in the **scaled** space. To plot them on the original scale we must reverse the scaling with `scaler.inverse_transform()`.

```python
# Guide:
kmeans_final = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
df['segment'] = kmeans_final.fit_predict(X_scaled)

print(f"Inertia: {kmeans_final.inertia_:.2f}")
print(df['segment'].value_counts().sort_index())
```

### What to do
1. Fit the final model using the best k from Steps 5 and 6
2. Store the cluster labels in a new column `df['segment']`
3. Print the inertia
4. Print how many members are in each segment

In [ ]:
# 1 & 2. Fit the final model and store labels
# YOUR CODE HERE
kmeans_final = ...
df['segment'] = ...


# 3. Print inertia
# YOUR CODE HERE


# 4. Print member count per segment
# YOUR CODE HERE


**Reflection:** Are the segments roughly equal in size? What might it mean for the business if one segment is much larger than the others?

*Your answer:*

## Step 8 - Visualise the Clusters

### Key Concept
A scatter plot coloured by cluster label makes the segmentation immediately visible. We also plot the **centroids** — the centre of each cluster — so it is clear what the "typical" member in each group looks like.

Because we scaled the data before clustering, the centroids are in scaled space. We need to convert them back to original units using `scaler.inverse_transform()` before plotting.

```python
# Guide — converting centroids back to original scale:
centroids_original = scaler.inverse_transform(kmeans_final.cluster_centers_)

# Guide — plotting each cluster separately:
colours = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for seg in range(best_k):
    mask = df['segment'] == seg
    plt.scatter(df.loc[mask, 'visits_per_month'],
                df.loc[mask, 'avg_session_minutes'],
                c=colours[seg], label=f'Segment {seg}',
                alpha=0.7, edgecolors='white', s=60)

# Plot centroids on top:
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            c='black', marker='X', s=250, zorder=5, label='Centroids')
```

### What to do
Create a scatter plot that shows all members coloured by their segment, with centroids marked in black.

In [ ]:
# Convert centroids back to original scale
# YOUR CODE HERE
centroids_original = ...


# Create the scatter plot coloured by segment
colours = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

plt.figure(figsize=(9, 6))

# YOUR CODE HERE — loop through segments and plot each one


# YOUR CODE HERE — plot the centroids


plt.xlabel('Visits per Month', fontsize=12)
plt.ylabel('Avg Session Minutes', fontsize=12)
plt.title('Gym Member Segments — K-Means', fontsize=13, fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Reflection:** Are the clusters clearly separated in the chart? Do any clusters overlap? What does overlap between clusters suggest about the data?

*Your answer:*

## Step 9 - Summarise and Label Each Segment

### Key Concept
Cluster numbers (0, 1, 2 …) have no meaning on their own. The most important step in any clustering project is **interpreting** what each cluster actually represents.

We do this by:
1. Computing the **average values** of each feature per cluster
2. Using those averages to give each cluster a **descriptive business name**

```python
# Guide — compute average features per segment:
summary = df.groupby('segment')[['visits_per_month', 'avg_session_minutes']].mean().round(1)
summary['count'] = df.groupby('segment')['segment'].count()
print(summary)
```

### What to do
1. Compute the mean of both features per segment and count members — store in `summary`
2. Print `summary` and examine the values
3. Based on the means, assign a descriptive label to each segment and add it as a new column `summary['label']`
4. Plot two side-by-side horizontal bar charts: one for average visits, one for average session minutes

```python
# Guide — mapping labels:
# Look at your summary table, then manually decide which segment is which.
# Example (your numbers may differ):
labels_map = {
    0: 'Casual Visitors',
    1: 'Social Members',
    2: 'Dedicated Trainers',
    3: 'Elite Members'
}
summary['label'] = summary.index.map(labels_map)
```

In [ ]:
# 1 & 2. Compute segment summary and print it
# YOUR CODE HERE
summary = ...
summary['count'] = ...

print(summary)

In [ ]:
# 3. Assign descriptive labels — edit the dictionary to match your summary values
# YOUR CODE HERE
labels_map = {
    0: '???',
    1: '???',
    2: '???',
    3: '???'
}
summary['label'] = summary.index.map(labels_map)
print(summary[['label', 'visits_per_month', 'avg_session_minutes', 'count']])

In [ ]:
# 4. Two side-by-side horizontal bar charts
# YOUR CODE HERE
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: average visits per month
# axes[0].barh(summary['label'], summary['visits_per_month'], color=colours)
# YOUR CODE HERE


# Right: average session minutes
# YOUR CODE HERE


plt.tight_layout()
plt.show()

**Reflection:** Does the summary table match the groups you expected from the raw scatter plot in Step 3? Were there any surprises?

*Your answer:*

---
## Step 10 — Reflect and Interpret

### Key Concept
Clustering is only useful if the results lead to **actionable decisions**. In this final step you will translate the segments into concrete business recommendations and also think critically about the limitations of the approach.

### What to do
Answer the four questions below in your own words, then run the final summary code cell.

**Question 1 — Business Action**

For each of the four segments, suggest one specific marketing action the gym could take:

| Segment | Suggested Action |
|---------|------------------|
| Casual Visitors | ??? |
| Social Members | ??? |
| Dedicated Trainers | ??? |
| Elite Members | ??? |

**Question 2 — Scaling**

What would have happened if we had skipped the scaling step in Step 4? Which feature would have dominated and why?

*Your answer:*

**Question 3 — Limitations**

K-means assumes clusters are roughly circular and roughly equal in size. Looking at your scatter plot from Step 8, do the clusters appear to meet these assumptions? Are there any members that look like they could belong to more than one cluster?

*Your answer:*

**Question 4 — Extending the Analysis**

The current model only uses two features. Name two additional features about gym members that might improve the segmentation and explain why.

*Your answer:*

In [ ]:
# Final summary — run this cell to print your complete pipeline results
print("=" * 55)
print("  GYM MEMBER SEGMENTATION — PIPELINE SUMMARY")
print("=" * 55)
print(f"  Total members      : {len(df)}")
print(f"  Features used      : visits_per_month, avg_session_minutes")
print(f"  Optimal k          : {best_k}")
print(f"  Final WCSS         : {kmeans_final.inertia_:.2f}")
print(f"  Silhouette score   : {silhouette_score(X_scaled, df['segment']):.4f}")
print()
print("  Segment Summary:")
for _, row in summary.iterrows():
    print(f"    {row['label']:<22}: {int(row['count']):3d} members  "
          f"| {row['visits_per_month']:.1f} visits/mo  "
          f"| {row['avg_session_minutes']:.0f} min/session")
print("=" * 55)